# V4 Universal Football Model - Footballdata.io Ingestion

This notebook tests the Footballdata.io API and prepares data for the V4 model.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")
if not API_KEY:
    print("API key not found. Please check ../.env.")
else:
    print(f"API key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

## 1. Basic API Fetch Function

In [ ]:
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Fetch a JSON payload from Footballdata.io."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    response = requests.get(url, headers=headers, params=params or {})
    response.raise_for_status()
    return response.json()

## 2. Test Connection

In [ ]:
data = fetch_footballdata("leagues")
print(json.dumps(data, indent=2)[:500] + "\n...[truncated]")

## 3. Inspecting Match Endpoints
Inspect the response shape before selecting a match ID, since `data` may be a list or dictionary.

In [ ]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")
match_id_to_test = None

if today_data:
    print(f"Top-level keys: {list(today_data.keys())}")
    payload = today_data.get("data")
    if isinstance(payload, dict):
        print(f"Data keys: {list(payload.keys())}")
        fixtures = payload.get("fixtures", payload.get("matches", []))
    elif isinstance(payload, list):
        fixtures = payload
    else:
        fixtures = []
    if fixtures:
        match_id_to_test = fixtures[0].get("match_id")
        print(f"Using match_id: {match_id_to_test}")
    else:
        print("No fixtures found in the response.")
else:
    print("Failed to fetch /fixtures/today.")

## 4. Fetching Granular Match Data

In [ ]:
stats_data = None
events_data = None
if match_id_to_test:
    stats_data = fetch_footballdata(f"matches/{match_id_to_test}/stats")
    events_data = fetch_footballdata(f"matches/{match_id_to_test}/events")
    print("Stats:\n", json.dumps(stats_data, indent=2)[:500])
    print("Events:\n", json.dumps(events_data, indent=2)[:500])
else:
    print("No match ID available for detailed endpoint tests.")

## 5. V4 Ingestion Parser

In [ ]:
def parse_footballdata_to_v4(match_info, events_payload, stats_payload):
    """Transform Footballdata.io payloads into the V4 schema."""
    parsed = {
        "home_team": match_info.get("home_team_name"),
        "away_team": match_info.get("away_team_name"),
        "home_score": match_info.get("home_score", 0),
        "away_score": match_info.get("away_score", 0),
        "current_minute": match_info.get("minute", 0),
        "red_cards": {"home": 0, "away": 0},
        "live_xg": {"home": 0.0, "away": 0.0},
        "starting_xi": {"home": [], "away": []}
    }
    if events_payload and events_payload.get("success"):
        for event in events_payload.get("data", []):
            if event.get("type") == "Red Card":
                side = "home" if event.get("team") == "home" else "away"
                parsed["red_cards"][side] += 1
    return parsed